<a href="https://colab.research.google.com/github/devisheshamalinis2449sse-sudo/Machine-learning/blob/main/ML_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ==========================================
# SMART INDUSTRIAL FAULT PREDICTION SYSTEM
# Bayesian Network + Deep Neural Network
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

# ----------------------------------
# LOAD CSV DATASET
# ----------------------------------

df = pd.read_csv("/content/Smart_Industrial_Fault_Prediction_100_Columns (1).csv")

print(df.head())

# ----------------------------------
# ENCODE TARGET LABEL
# ----------------------------------

le = LabelEncoder()
# Corrected column name from 'Fault_Type' to 'Fault_Status'
df['Fault_Status'] = le.fit_transform(df['Fault_Status'])

# Removed 'Machine_ID' as it's not present in the df head, and corrected 'Fault_Type' to 'Fault_Status'
X = df.drop(['Fault_Status'], axis=1)
y = df['Fault_Status']

# ----------------------------------
# TRAIN TEST SPLIT
# ----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ----------------------------------
# FEATURE SCALING
# ----------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==================================
# MODULE 1 : BAYESIAN NETWORK
# ==================================

print("\nBayesian Fault Probability Estimation")

bayes_model = GaussianNB()

bayes_model.fit(X_train, y_train)

bayes_pred = bayes_model.predict(X_test)

print("Bayesian Accuracy:",
      accuracy_score(y_test, bayes_pred))

print(classification_report(y_test, bayes_pred))

# Fault probabilities
fault_probability = bayes_model.predict_proba(X_test)

print("\nSample Fault Probabilities:")
print(fault_probability[:5])

# ==================================
# MODULE 2 : DEEP NEURAL NETWORK
# ==================================

print("\nDeep Neural Fault Classification")

num_classes = len(np.unique(y))

y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

model = Sequential()

model.add(Dense(64,
                activation='relu',
                input_shape=(X_train.shape[1],)))

model.add(Dense(32,
                activation='relu'))

model.add(Dense(16,
                activation='relu'))

model.add(Dense(num_classes,
                activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_cat,
    epochs=50,
    batch_size=16,
    validation_split=0.2
)

loss, accuracy = model.evaluate(
    X_test,
    y_test_cat
)

print("\nDNN Accuracy:", accuracy)

# Predictions
predictions = model.predict(X_test)

predicted_classes = np.argmax(
    predictions,
    axis=1
)

print("\nClassification Report")
print(classification_report(
    y_test,
    predicted_classes
))

# ==================================
# TEST WITH NEW SENSOR DATA
# ==================================

# Corrected new_data to match the number of features (X_train.shape[1])
# Assuming 100 sensor features, filling with zeros as placeholder.
# You should replace these zeros with actual sensor readings for a meaningful test.
new_data = np.zeros((1, X_train.shape[1])) # Creates an array with 1 row and 100 columns filled with zeros
# Example: if you wanted specific values for the first few sensors, you could do:
# new_data[0, :8] = [95, 8.5, 92, 60, 215, 12.8, 80, 1600]

new_data = scaler.transform(new_data)

# Bayesian probability
bayes_prob = bayes_model.predict_proba(new_data)

# DNN classification
dnn_pred = model.predict(new_data)

print("\nBayesian Fault Probability:")
print(bayes_prob)

fault_class = np.argmax(dnn_pred)

print("\nPredicted Fault Type:")
print(le.inverse_transform([fault_class])[0])

   Sensor_1  Sensor_2  Sensor_3  Sensor_4  Sensor_5  Sensor_6  Sensor_7  \
0     67.55     12.25     34.75     30.09     76.28     70.90     90.30   
1     11.03     74.86     71.35     58.33     34.01     67.69     20.04   
2     97.21     93.37     86.38     24.97     53.71     29.24     46.09   
3     91.35     59.10     85.11     62.43     23.33     21.47     37.74   
4     70.81     56.01     81.44     96.38     76.24     69.30     35.54   

   Sensor_8  Sensor_9  Sensor_10  ...  Sensor_92  Sensor_93  Sensor_94  \
0     17.82     47.97      12.68  ...      66.47      81.29      47.99   
1     49.13     50.84      95.84  ...      35.07      32.48      93.09   
2     15.28     44.11      98.68  ...      79.40      67.34      33.58   
3     90.91     81.65      87.46  ...      65.40      37.09      59.31   
4     69.75     65.73      18.40  ...      34.56      53.71      45.00   

   Sensor_95  Sensor_96  Sensor_97  Sensor_98  Sensor_99  Sensor_100  \
0      15.72      44.35      99.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/keras/src

8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.0859 - loss: 1.8544 - val_accuracy: 0.1875 - val_loss: 1.6277
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1484 - loss: 1.5655 - val_accuracy: 0.2812 - val_loss: 1.5282
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4609 - loss: 1.3957 - val_accuracy: 0.5000 - val_loss: 1.4667
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6797 - loss: 1.2599 - val_accuracy: 0.5312 - val_loss: 1.4302
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7500 - loss: 1.1282 - val_accuracy: 0.5000 - val_loss: 1.4095
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7344 - loss: 0.9982 - val_accuracy: 0.5000 - val_loss: 1.4061
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7266 - loss: 0.8749 - val_accuracy: 0.5000 - val_loss: 1.4130
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7344 - loss: 0.7670 - val_accuracy: 0.5000 - val_loss: 1.4233
Epoch 9/50


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
